In [2]:
import torch as t
from torch import nn
import torch.nn.functional as F 
import math 
import numpy 

In [ ]:
class MultiHeadAttention(nn.Module):
    def __init__(self , d_model , n_heads):
        super().__init__()
        self.d_model = d_model 
        self.n_heads = n_heads
        self.d_k =d_model // n_heads

        
        self.Wq = nn.Linear(d_model , self.d_k*n_heads)
        self.Wk = nn.Linear(d_model , self.d_k*n_heads)
        self.Wv = nn.Linear(d_model , self.d_k*n_heads)
        self.fc = nn.Linear(self.d_k * n_heads, d_model)
    
    def forward(self, x):
        N , T , _ = x.shape
        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)
        
        
        Q = Q.view(N, T, self.n_heads, self.d_k).transpose(1, 2)
        K = K.view(N, T, self.n_heads, self.d_k).transpose(1, 2)
        V = V.view(N, T, self.n_heads, self.d_k).transpose(1, 2)
        #score
        # Note : dont use K.T becuase it is 3 dim not 2d (B , T , d_model)
        score = (Q@K.transpose(-2,-1))/(t.sqrt( t.tensor(self.d_k , dtype=t.float32)))
        weight = F.softmax(score , dim=-1) # softmax needs float
        
        A = weight@V
        A = A.transpose(1,2).contiguous().view(N , T , self.d_k*self.n_heads)
        
        return self.fc(A)